### Hyperparameter optimization with OptunaSearches the DeepONet and Closure-DeepONet hyperparameters under a chosen splitting protocol. **Why is the objective the validation partition?** Under the grouped protocol the validation cases are disjoint from both the training and the test cases, so no test simulation influences the search. The winning configuration is retrained once and scored on the untouched test cases. Tuning against the test split would reintroduce exactly the leakage the protocol exists to remove — say this explicitly in the methods section, because a reviewer will ask.Put `co2gcs/` and `All_dataset.csv` next to this notebook.

In [1]:
# !pip install optuna

import os
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
import torch

import co2gcs as C
from co2gcs import tune as T
from co2gcs.physics import violation_rate

torch.set_num_threads(1)  # Keep 1 thread per trial on CPU

optuna.logging.set_verbosity(optuna.logging.WARNING)

C.plots.use_style()

os.makedirs('tuning', exist_ok=True)

### 1. Choose the run

In [2]:
# ---- Optuna configuration --------------------------------------------------

ARCH = 'closure'          # 'closure' | 'baseline'
PROTOCOL = 'group'       # 'group' | 'random'
MODE = 'fixed'           # 'fixed' | 'orig'
SPACE = 'extended'          # 'basic' (7 params) | 'extended' (13–14 params)

N_TRIALS = 100            # Add more trials by re-running the optimization cell
SEARCH_EPOCHS = 200       # Short budget per trial; winner is retrained in full
SEED = 42

print(T.FIXED_NOTE[SPACE][ARCH])

Extended HPO: basic parameters plus dropout, KL weight, epochs, patience, variance weighting, physics weight and n_freq.


In [3]:
CUSTOM_SPACE = {
    "closure": {

        "lr": {"type": "float", "low": 1e-5,
            "high": 1e-2, "log": True
        },

        "wd": {"type": "float", "low": 1e-6,
            "high": 1e-3, "log": True
        },

        "bs": {"type": "categorical",
            "choices": [32, 64, 128, 512, 1024]
        },

        "width": {"type": "categorical",
            "choices": [128, 192, 256]
        },

        "n_branch_layers": {"type": "int",
            "low": 2, "high": 5
        },

        "n_trunk_layers": {"type": "int",
            "low": 4, "high": 10
        },

        "p": {
            "type": "categorical",
            "choices": [32, 64, 128, 192]
        },

        "sigma": {"type": "float", "low": 0.1, 
                  "high": 1.0, "log": True
        },

        "p_drop": {"type": "float",
            "low": 0.0,  "high": 0.20
        },

        "kl_weight": {"type": "float", "low": 1e-3,  
            "high": 0.1, "log": True
        },

        "epochs": {"type": "int", "low": 100,
            "high": 500, "step": 100
        },

        "patience": {"type": "int", "low": 20,
            "high": 50, "step": 5
        },

        "var_weight": {"type": "categorical",
            "choices": [True]
        },

        "phys_weight": {"type": "float", "low": 1e-4,
            "high": 0.1, "log": True
        },

        "n_freq": {"type": "categorical",
            "choices": [16, 32, 64]
        },
    },

    "baseline": {

        "lr": {"type": "float", "low": 1e-5,
            "high": 1e-2, "log": True
        },

        "wd": {"type": "float", "low": 1e-6,
            "high": 1e-3, "log": True
        },

        "bs": {
            "type": "categorical",
            "choices": [32, 64, 128, 512, 1024]
        },

        "width": {
            "type": "categorical",
            "choices": [96, 128, 192]
        },

        "n_branch_layers": {"type": "int",
            "low": 1, "high": 4
        },

        "n_trunk_layers": {"type": "int",
            "low": 5, "high": 10
        },

        "closure_pen": {"type": "float", "low": 1e-2, 
            "high": 10.0, "log": True
        },

        "p_drop": {"type": "float",
            "low": 0.0, "high": 0.20
        },

        "kl_weight": {"type": "float", "low": 1e-3,
            "high": 0.1, "log": True
        },

        "epochs": {"type": "int",       
            "low": 100, "high": 500, "step": 100
        },

        "patience": {"type": "int",
            "low": 20, "high": 50, "step": 5
        },

        "var_weight": {"type": "categorical",
            "choices": [True]
        },

        "phys_weight": {"type": "float",
            "low": 1e-4, "high": 0.1, "log": True
        },
    }
}

### 2. What is searched**basic** — the seven parameters that move the grouped-protocol score most.| Closure | Baseline ||---|---|| learning rate 1e-4 – 5e-3 | learning rate 1e-4 – 5e-3 || weight decay 1e-7 – 1e-3 | weight decay 1e-7 – 1e-3 || width 128 / 192 / 256 | width 96 / 128 / 192 || n_layers 3 – 5 | n_branch 1 – 3 || basis dim p 64 – 256 | n_trunk 4 – 9 || Fourier sigma 0.1 – 2.0 | batch size 1024 – 4096 || batch size 512 – 2048 | closure penalty 1e-2 – 1e1 |**extended** — adds dropout (0–0.30), KL weight (1e-3 – 1), epochs (60–220),patience (15–50), variance weighting (on/off), physics-loss weight(1e-4 – 1) and, for the closure net, n_freq (16/32/64). Budget more trials:epochs and patience interact with the pruner, and `var_weight=False` reproducesthe MTI collapse in some trials — informative, but it costs trials.

In [4]:
# ---- Preview Optuna search space (no training) -----------------------------

study_preview = optuna.create_study(direction='maximize')

trial_preview = study_preview.ask()

# Draw one example from the basic search space
hp = T.suggest_basic(trial_preview, ARCH)

# Add extended parameters if requested
if SPACE == 'extended':
    hp = T.suggest_extended(trial_preview, ARCH, hp)

# Display the example hyperparameter configuration
pd.Series(hp).to_frame('example draw')

,example draw
lr,0.000178
wd,0.000002
bs,2048
width,256
n_branch_layers,4
n_trunk_layers,6
p,64
sigma,1.741214
p_drop,0.103622
kl_weight,0.013263


### 3. Run the studyThe study is stored in SQLite under `optuna/`, so **re-running this cell addstrials** rather than starting over — the practical way to work in a notebook,since each trial trains a network. Median pruning kills unpromising trials onthe validation curve.On CPU expect roughly 45–90 s per closure trial and 2–4 min per baseline trialat `SEARCH_EPOCHS=45`. Start with 10–15 trials, look at the results, then addmore.

In [ ]:
# ---- Run Optuna hyperparameter optimization -------------------------------

study, d = T.run_study(
    ARCH,
    PROTOCOL,
    mode=MODE,
    n_trials=N_TRIALS,
    search_epochs=SEARCH_EPOCHS,
    seed=SEED,
    space=SPACE,
    search_space=CUSTOM_SPACE
)

# Separate completed and pruned trials
done = [
    t for t in study.trials
    if t.value is not None
]

pruned = [
    t for t in study.trials
    if str(t.state) == 'TrialState.PRUNED'
]

print(
    f'{len(study.trials)} trials total: '
    f'{len(done)} complete, {len(pruned)} pruned'
)

# Report best trial
if done:
    print(
        f'Best validation R²_mean = {study.best_value:.4f}'
    )

    print('\nBest hyperparameters:')
    display(
        pd.Series(study.best_params)
        .to_frame('best value')
    )
else:
    print('No completed Optuna trials.')

### Optional: a progress bar and a wall-clock cap`timeout` is often more convenient than `n_trials` in a notebook — run for afixed number of minutes and stop wherever it gets to.

# ---- Extended Optuna optimization ------------------------------------------

study, d = T.run_study(
    ARCH,
    PROTOCOL,
    mode=MODE,
    n_trials=1000,
    timeout=20 * 60,          # 20-minute maximum runtime
    search_epochs=SEARCH_EPOCHS,
    seed=SEED,
    space=SPACE,
    search_space=CUSTOM_SPACE
)

### 4. Trials table and parameter importance

In [ ]:
# ---- Optuna trial history --------------------------------------------------

hist = pd.DataFrame([
    {
        **t.params,
        'value': t.value,
        'state': str(t.state).split('.')[-1]
    }
    for t in study.trials
])

# Keep only completed trials with a valid objective value
completed = hist[
    (hist['state'] == 'COMPLETE') &
    (hist['value'].notna())
].copy()

# Show the 10 best hyperparameter configurations
top10 = completed.sort_values(
    'value',
    ascending=False
).head(10)

display(top10)

top10 = top10.rename(
    columns={'value': 'val_R2_mean'}
)

display(top10.round(6))

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    completed.index,
    completed['value'],
    marker='o',
    linewidth=1
)

plt.xlabel('Trial')
plt.ylabel('Validation R²_mean')
plt.title('Optuna optimization history')
plt.grid(alpha=0.3)

plt.show()

In [ ]:
completed_plot = completed.reset_index(drop=True)

best_so_far = completed_plot['value'].cummax()

plt.figure(figsize=(8, 5))

plt.plot(
    completed_plot.index + 1,
    best_so_far,
    linewidth=2
)

plt.xlabel('Completed trial')
plt.ylabel('Best validation R²_mean')
plt.title('Optuna best-so-far validation performance')
plt.grid(alpha=0.3)

plt.show()

In [ ]:
# ---- Optuna hyperparameter importance -------------------------------------

imp = T.importance(study)

if imp:
    s = pd.Series(imp).sort_values()

    fig, ax = plt.subplots(
        figsize=(7, 0.4 * len(s) + 1.5)
    )

    ax.barh(
        s.index,
        s.values,
        height=0.68
    )

    ax.set_xlabel('fANOVA importance')
    ax.set_ylabel('Hyperparameter')

    ax.grid(
        axis='y',
        visible=False
    )

    ax.set_title(
        f'{ARCH.capitalize()} DeepONet / {PROTOCOL} split: '
        'hyperparameter importance',
        fontsize=11
    )

    plt.tight_layout()
    plt.show()

    # Numerical ranking
    importance_table = (
        s.sort_values(ascending=False)
        .to_frame('importance')
        .round(3)
    )

    display(importance_table)

else:
    print('Need more completed trials for importance analysis.')

Importances shift a lot below ~30 trials. Treat a 15-trial ranking as adirection, not a result to quote.

In [ ]:
# ---- Optuna optimization history + most important parameter ---------------

# Keep only completed trials with valid objective values
completed_trials = [
    t for t in study.trials
    if t.value is not None
]

v = np.array([
    t.value for t in completed_trials
])

fig, ax = plt.subplots(
    1, 2,
    figsize=(10, 3.8)
)

# ---------------------------------------------------------------------------
# 1. Optimization history
# ---------------------------------------------------------------------------
trial_numbers = np.arange(1, len(v) + 1)

ax[0].plot(
    trial_numbers,
    v,
    'o',
    ms=4,
    alpha=0.7
)

ax[0].plot(
    trial_numbers,
    np.maximum.accumulate(v),
    '-',
    lw=1.6,
    label='Best so far'
)

ax[0].set_xlabel('Completed trial')
ax[0].set_ylabel(r'Validation $R^2_{\mathrm{mean}}$')
ax[0].legend(fontsize=8)
ax[0].set_title(
    'Optuna optimization history',
    fontsize=10
)

# ---------------------------------------------------------------------------
# 2. Most important hyperparameter
# ---------------------------------------------------------------------------
top = (
    max(imp, key=imp.get)
    if imp
    else list(study.best_params)[0]
)

# Build parameter/value table directly from completed trials
sub = pd.DataFrame([
    {
        top: t.params.get(top),
        'value': t.value
    }
    for t in completed_trials
]).dropna()

ax[1].scatter(
    sub[top],
    sub['value'],
    s=26,
    alpha=0.75
)

ax[1].set_xlabel(top)
ax[1].set_ylabel(r'Validation $R^2_{\mathrm{mean}}$')

# Log scale when appropriate
if (
    pd.api.types.is_numeric_dtype(sub[top])
    and (sub[top] > 0).all()
    and sub[top].nunique() > 3
):
    ax[1].set_xscale('log')

ax[1].set_title(
    f'Effect of most important parameter: {top}',
    fontsize=10
)

plt.tight_layout()
plt.show()

### 5. Retrain the winner and score it on untouched test casesThree things are reported besides accuracy:* **closure MAE** — exact zero for the closure architecture, non-zero for the baseline.* **TV ratio** — predicted trajectory total variation divided by the simulations'.  Near 1 is good; values well above 1 mean the model oscillates between stored  timesteps, which pointwise $R^2$ cannot see.* **physics violation rate** — fraction of constrained (row, index) pairs whose  predicted time-derivative has the wrong sign, using the regime-aware  constraint set in `co2gcs.physics`.

In [ ]:
# ---- Default vs. tuned model comparison -----------------------------------

from co2gcs.train import predict_mean


def tv_ratio(model, d):
    """
    Temporal variation ratio between predicted and true trapping-index
    trajectories.

    A value close to 1 indicates that the model reproduces the temporal
    variation of the reference trajectories well.
    """
    P = predict_mean(
        model,
        d['Xb_te'],
        d['Xt_te']
    )

    y = d['y_te']
    g = d['groups_te']
    t = d['t_te']

    pv = []
    yv = []

    for c in np.unique(g):
        m = g == c
        order = np.argsort(t[m])

        pv.append(
            np.abs(
                np.diff(P[m][order], axis=0)
            ).sum(axis=0)
        )

        yv.append(
            np.abs(
                np.diff(y[m][order], axis=0)
            ).sum(axis=0)
        )

    pv = np.asarray(pv)
    yv = np.asarray(yv)

    return float(
        np.mean(
            np.median(pv, axis=0)
            / np.maximum(
                np.median(yv, axis=0),
                1e-9
            )
        )
    )


# ---------------------------------------------------------------------------
# Fit and evaluate default and tuned configurations
# ---------------------------------------------------------------------------

rows = {}

configs = {
    'default': T.DEFAULTS[ARCH],
    'tuned': {
        **T.DEFAULTS[ARCH],
        **study.best_params
    }
}

for tag, hp in configs.items():

    model, met, _ = T.final_fit(
        ARCH,
        hp,
        d
    )

    # Physics / constraint violations
    vr = violation_rate(
        model,
        d['Xb_te'],
        d['Xt_te'],
        d['Xt_plus_te'],
        d['phys_sign_te']
    )

    # Store results
    rows[tag] = {
        'R2_mean': met['R2_mean'],

        **{
            f'R2_{k}': met[f'R2_{k}']
            for k in C.TARGETS
        },

        'RMSE_mean': met['RMSE_mean'],
        'MAE_mean': met['MAE_mean'],
        'closure_MAE': met['closure_MAE'],

        'TV_ratio': tv_ratio(model, d),

        **{
            f'viol_{k}': v
            for k, v in vr.items()
        }
    }

    print(
        f'{tag:8s} '
        f'R2_mean={met["R2_mean"]:.4f}  '
        f'RMSE={met["RMSE_mean"]:.4f}  '
        f'MAE={met["MAE_mean"]:.4f}  '
        f'closure={met["closure_MAE"]:.4f}'
    )


# ---------------------------------------------------------------------------
# Comparison table
# ---------------------------------------------------------------------------

cmp = pd.DataFrame(rows).T

display(
    cmp.round(4)
)

# ---------------------------------------------------------------------------
# Improvement from tuning
# ---------------------------------------------------------------------------

delta_r2 = (
    cmp.loc['tuned', 'R2_mean']
    - cmp.loc['default', 'R2_mean']
)

delta_rmse = (
    cmp.loc['tuned', 'RMSE_mean']
    - cmp.loc['default', 'RMSE_mean']
)

delta_mae = (
    cmp.loc['tuned', 'MAE_mean']
    - cmp.loc['default', 'MAE_mean']
)

print(
    f'\nTuned - default: '
    f'ΔR²_mean = {delta_r2:+.4f}, '
    f'ΔRMSE = {delta_rmse:+.4f}, '
    f'ΔMAE = {delta_mae:+.4f}'
)

### 6. Save the winning configuration for the other notebooks

In [ ]:
# ---- Save best configuration and comparison results -----------------------

best = {
    **T.DEFAULTS[ARCH],
    **study.best_params
}

# File containing the best Optuna configuration
path = (
    f'tuning/best_{ARCH}_{PROTOCOL}_{MODE}_{SPACE}.json'
)

# Build export dictionary
export = {
    'arch': ARCH,
    'protocol': PROTOCOL,
    'mode': MODE,
    'space': SPACE,

    'best_params': study.best_params,

    'config': best,

    'best_val_r2': float(study.best_value),

    'n_trials': len(study.trials),

    'test': {
        k: {
            kk: float(vv)
            for kk, vv in v.items()
        }
        for k, v in rows.items()
    }
}

# Save JSON
with open(path, 'w') as f:
    json.dump(
        export,
        f,
        indent=2
    )

# Save default-vs-tuned comparison table
cmp_path = (
    f'tuning/compare_{ARCH}_{PROTOCOL}_{MODE}_{SPACE}.csv'
)

cmp.to_csv(
    cmp_path,
    index=True
)

print('Wrote:')
print(f'  Configuration : {path}')
print(f'  Comparison    : {cmp_path}')

### 7. Both architectures, both protocolsEach study is independent and separately resumable. Run this over severalsessions rather than in one sitting — it is a lot of network training.

In [ ]:
# ---- Quick Optuna comparison: architecture × protocol ---------------------

for arch in ['closure', 'baseline']:

    for proto in ['group', 'random']:

        st, dd = T.run_study(
            arch,
            proto,
            mode=MODE,
            n_trials=10,
            search_epochs=SEARCH_EPOCHS,
            seed=SEED,
            space=SPACE
        )

        print(
            f'{arch:9s} {proto:7s} '
            f'best val R² = {st.best_value:.4f} '
            f'({len(st.trials)} trials)'
        )

In [ ]:
# ---- Quick Optuna comparison with results table ---------------------------

quick_results = []

for arch in ['closure', 'baseline']:

    for proto in ['group', 'random']:

        st, dd = T.run_study(
            arch,
            proto,
            mode=MODE,
            n_trials=10,
            search_epochs=SEARCH_EPOCHS,
            seed=SEED,
            space=SPACE
        )

        quick_results.append({
            'architecture': arch,
            'protocol': proto,
            'best_val_R2': st.best_value,
            'n_trials': len(st.trials)
        })

        print(
            f'{arch:9s} {proto:7s} '
            f'best val R² = {st.best_value:.4f} '
            f'({len(st.trials)} trials)'
        )

quick_results = pd.DataFrame(quick_results)

print('\nOptuna comparison:')
display(
    quick_results
    .sort_values('best_val_R2', ascending=False)
    .round(4)
)

## Reading the resultReport the search protocol, not just the winning numbers: search space, number oftrials, pruning rule, the partition the objective was computed on, and the factthat the test cases were touched exactly once. If the tuned gain is small — itoften is here — that is worth stating plainly. A modest gain over a sensibledefault is evidence the architecture is robust, which is a better story than anunexplained jump.